In [2]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import sys
sys.path.append("../../../../LoRO")

from utils import *

In [3]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from datasets import load_dataset
import tqdm

tokenizer = AutoTokenizer.from_pretrained("hyunwoongko/roberta-base-en-mnli")
model = AutoModelForSequenceClassification.from_pretrained("hyunwoongko/roberta-base-en-mnli")

/home/ubuntu/anaconda3/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/home/ubuntu/anaconda3/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [4]:
print(model)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [5]:
model = model_obfuscation(model)

Obfuscating: roberta.encoder.layer.0.attention.self.query
Obfuscating: roberta.encoder.layer.0.attention.self.key
Obfuscating: roberta.encoder.layer.0.attention.self.value
Obfuscating: roberta.encoder.layer.0.attention.output.dense
Obfuscating: roberta.encoder.layer.0.intermediate.dense
Obfuscating: roberta.encoder.layer.0.output.dense
Obfuscating: roberta.encoder.layer.1.attention.self.query
Obfuscating: roberta.encoder.layer.1.attention.self.key
Obfuscating: roberta.encoder.layer.1.attention.self.value
Obfuscating: roberta.encoder.layer.1.attention.output.dense
Obfuscating: roberta.encoder.layer.1.intermediate.dense
Obfuscating: roberta.encoder.layer.1.output.dense
Obfuscating: roberta.encoder.layer.2.attention.self.query
Obfuscating: roberta.encoder.layer.2.attention.self.key
Obfuscating: roberta.encoder.layer.2.attention.self.value
Obfuscating: roberta.encoder.layer.2.attention.output.dense
Obfuscating: roberta.encoder.layer.2.intermediate.dense
Obfuscating: roberta.encoder.layer.2

In [6]:
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device='cuda')

In [7]:
# two examples from training set

sentence = "Conceptually cream skimming has two basic dimensions - product and geography. Product and geography are what make cream skimming work." # 2 neutral

print(classifier(sentence))

sentence = "you know during the season and i guess at at your level uh you lose them to the next level if if they decide to recall the the parent team the Braves decide to call to recall a guy from triple A then a double A guy goes up to replace him and a single A guy goes up to replace him. You lose the things to the following level if the people recall." # 1 entailment

print(classifier(sentence))

sentence = "At the end of Rue des Francs-Bourgeois is what many consider to be the city's most handsome residential square, the Place des Vosges, with its stone and red brick facades. Place des Vosges is constructed entirely of gray marble." # 0 contradiction
print(classifier(sentence))

[{'label': 'LABEL_2', 'score': 0.9980720281600952}]
[{'label': 'LABEL_1', 'score': 0.9928887486457825}]
[{'label': 'LABEL_0', 'score': 0.9991104006767273}]


In [8]:
dataset = load_dataset("glue", "mnli")['validation_matched']
print(dataset)

Dataset({
    features: ['premise', 'hypothesis', 'label', 'idx'],
    num_rows: 9815
})


In [9]:
correct = 0
total = 0

for i in tqdm.tqdm(range(9815)):
    result = classifier(dataset[i]['premise'] + ' ' + dataset[i]['hypothesis'])

    if result[0]['label'] == 'LABEL_2':
        result = 1
    elif result[0]['label'] == 'LABEL_1':
        result = 0
    else:
        result = 2
        
    if result == dataset[i]['label']:
        correct += 1
    total += 1

print("correct:{}, total:{}, accuracy:{}".format(correct, total, correct/total))

100%|██████████| 9815/9815 [01:44<00:00, 94.04it/s]

correct:8375, total:9815, accuracy:0.8532857870606215
